<a href="https://colab.research.google.com/github/ofir2207/Cloud-project/blob/main/ex10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install Java and Spark
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.4.1/spark-3.4.1-bin-hadoop3.tgz
!tar xf spark-3.4.1-bin-hadoop3.tgz
!pip install -q findspark

In [ ]:
import os
import findspark

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.4.1-bin-hadoop3"
findspark.init()

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Big Data Example").getOrCreate()

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import json

FILE_PATH = "C:\Users\liraz\Downloads\json-20251229-2107.json"

with open(FILE_PATH, "r") as f:
    raw_data = json.load(f)

print(f"Loaded {len(raw_data)} records")
print("Sample record:", raw_data[0])

In [ ]:
rdd_raw = spark.sparkContext.parallelize(raw_data)

# Map: flatten each record into (parameter, value) pairs
rdd_pairs = rdd_raw.flatMap(
    lambda record: [(param, float(value))
                    for param, value in record.items()
                    if value is not None]
)

# Reduce: max and min per parameter
rdd_max = rdd_pairs.reduceByKey(lambda a, b: a if a > b else b)
rdd_min = rdd_pairs.reduceByKey(lambda a, b: a if a < b else b)

max_results = dict(rdd_max.collect())
min_results = dict(rdd_min.collect())

print("── Results ──")
for param in sorted(max_results.keys()):
    print(f"  {param:20s}  min = {min_results[param]:.2f}   max = {max_results[param]:.2f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

rdd_grouped = rdd_pairs.groupByKey().mapValues(list)
param_values = dict(rdd_grouped.collect())
params = sorted(param_values.keys())
n = len(params)

# Graph 1: Min vs Max bar chart
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(n)
ax.bar(x - 0.2, [min_results[p] for p in params], width=0.4, label="Min", color="steelblue")
ax.bar(x + 0.2, [max_results[p] for p in params], width=0.4, label="Max", color="tomato")
ax.set_xticks(x)
ax.set_xticklabels(params, rotation=30, ha="right")
ax.set_title("Min vs Max per Parameter")
ax.set_ylabel("Value")
ax.legend()
plt.tight_layout()
plt.show()

# Graph 2: Box plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.boxplot([param_values[p] for p in params], labels=params, patch_artist=True)
ax.set_title("Value Distribution per Parameter")
ax.set_ylabel("Value")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

# Graph 3: Line chart per parameter over time
fig, axes = plt.subplots(n, 1, figsize=(10, 3 * n), sharex=True)
if n == 1:
    axes = [axes]

for ax, param in zip(axes, params):
    ax.plot(param_values[param], color="mediumseagreen", linewidth=1)
    ax.axhline(min_results[param], color="steelblue", linestyle="--", linewidth=0.8, label=f"min={min_results[param]:.2f}")
    ax.axhline(max_results[param], color="tomato", linestyle="--", linewidth=0.8, label=f"max={max_results[param]:.2f}")
    ax.set_ylabel(param)
    ax.legend(loc="upper right", fontsize=8)

axes[-1].set_xlabel("Record index")
fig.suptitle("Parameter Values Over Time")
plt.tight_layout()
plt.show()

In [ ]:
spark.stop()